# NeMo Customizer Service Test (Refactored)

This notebook tests the NeMo Customizer service to verify it's working correctly.  
**Refactored version** with smaller cells for easier reading and editing.

## Prerequisites
- NeMo Customizer service deployed and running
- NeMo Data Store service deployed and running
- NeMo Entity Store service deployed and running
- Access to the cluster namespace

## Step 1: Install Required Packages

In [ ]:
%pip install requests python-dotenv huggingface_hub

## Step 2: Import Libraries and Configure URLs

In [ ]:
import os
import json
import time
import requests
from pathlib import Path
from pprint import pprint

# Load environment variables from env.donotcommit if it exists
try:
    from dotenv import load_dotenv
    env_donotcommit_path = Path.cwd() / "env.donotcommit"
    if env_donotcommit_path.exists():
        load_dotenv(env_donotcommit_path, override=False)
        print(f"✅ Loaded env.donotcommit from: {env_donotcommit_path}")
    else:
        print(f"ℹ️  env.donotcommit not found at: {env_donotcommit_path}")
        print(f"   Using environment variables from system/defaults")
except ImportError:
    print("⚠️  python-dotenv not installed - using system environment variables only")

In [ ]:
# Namespace and service URLs (cluster-internal)
NMS_NAMESPACE = os.getenv("NMS_NAMESPACE", "anemo-rhoai")
CUSTOMIZER_URL = f"http://nemocustomizer-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"
DATASTORE_URL = f"http://nemodatastore-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"
ENTITY_STORE_URL = f"http://nemoentitystore-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"

# API keys and tokens (from env.donotcommit)
NGC_API_KEY = os.getenv("NGC_API_KEY", "")
HF_TOKEN = os.getenv("HF_TOKEN", "")
NDS_TOKEN = os.getenv("NDS_TOKEN", "token")
NIM_SERVICE_ACCOUNT_TOKEN = os.getenv("NIM_SERVICE_ACCOUNT_TOKEN", "")

# Inference service (for testing models)
INFERENCE_SERVICE_URL = os.getenv("INFERENCE_SERVICE_URL", "")
INFERENCE_SERVICE_NAME = os.getenv("INFERENCE_SERVICE_NAME", "")
BASE_MODEL = os.getenv("BASE_MODEL", "meta/llama-3.2-1b-instruct")
CUSTOMIZATION_TEMPLATE = os.getenv("CUSTOMIZATION_TEMPLATE", "meta/llama-3.2-1b-instruct@v1.0.0")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HF_ENDPOINT"] = "https://huggingface.co"
    print("✅ Hugging Face token configured")
else:
    print("⚠️ HF_TOKEN not set - Hugging Face dataset access may be limited")

print(f"Customizer URL: {CUSTOMIZER_URL}")
print(f"DataStore URL: {DATASTORE_URL}")
print(f"Entity Store URL: {ENTITY_STORE_URL}")
print(f"Namespace: {NMS_NAMESPACE}")

## Step 2b: Helper Functions

Define reusable functions used later in the notebook. Run this cell once after configuring URLs (Step 2).

In [ ]:
import tempfile

def get_hf_api(datastore_url, nds_token="token"):
    """Return HfApi instance pointing at DataStore. Requires NMS_NAMESPACE, DATASTORE_URL in scope."""
    from huggingface_hub import HfApi
    endpoint = f"{datastore_url}/v1/hf"
    token = None if nds_token == "token" else nds_token
    return HfApi(endpoint=endpoint, token=token)

def ensure_datastore_namespace(hf_api, namespace):
    """Ensure namespace exists in DataStore (Gitea). Create temp repo then delete."""
    temp_repo_id = f"{namespace}/namespace-init-temp"
    try:
        hf_api.create_repo(repo_id=temp_repo_id, repo_type="dataset", exist_ok=True)
        try:
            hf_api.delete_repo(repo_id=temp_repo_id, repo_type="dataset")
        except Exception:
            pass
        print(f"✅ Namespace '{namespace}' initialized in Gitea")
    except Exception as e:
        print(f"ℹ️  Namespace check: {e}")

def create_or_reuse_dataset_repo(hf_api, repo_id, datastore_url):
    """Create dataset repo in DataStore if needed; reuse if accessible. Waits for git init."""
    check_url = f"{datastore_url}/v1/hf/api/datasets/{repo_id}/revision/main"
    try:
        r = requests.get(check_url, timeout=10)
        if r.status_code == 200:
            print(f"✅ Repo exists and is accessible - will reuse it")
            return
        if r.status_code == 500:
            print(f"⚠️  Repo corrupted (500) - will delete and recreate")
            try:
                hf_api.delete_repo(repo_id=repo_id, repo_type="dataset")
                print(f"   ✅ Deleted corrupted repo")
            except Exception as e:
                print(f"   ℹ️  Delete attempt: {e}")
    except Exception as e:
        print(f"ℹ️  Repo check failed: {e}")
    try:
        hf_api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
        print(f"✅ Created dataset repo: {repo_id}")
        print(f"   ⏳ Waiting 3s for git repository initialization...")
        time.sleep(3)
        print(f"✅ Repository ready for uploads")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"ℹ️ Dataset repo already exists: {repo_id}")
            time.sleep(3)
        else:
            raise

def upload_file_with_retry(hf_api, repo_id, local_path, path_in_repo, namespace, datastore_url, max_retries=2):
    """Upload a file to DataStore repo with retry on 500 (delete/recreate repo and retry).
    Pass local_path so the client can read the file and show upload progress (file object causes 0/0 LFS)."""
    for attempt in range(max_retries):
        try:
            hf_api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=path_in_repo,
                repo_id=repo_id,
                repo_type="dataset"
            )
            return True
        except Exception as e:
            err = str(e).lower()
            if ("500" in err or "internal server error" in err) and attempt < max_retries - 1:
                try:
                    hf_api.delete_repo(repo_id=repo_id, repo_type="dataset")
                    ensure_datastore_namespace(hf_api, namespace)
                    hf_api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
                    time.sleep(3)
                except Exception:
                    pass
            else:
                raise
    return False

def verify_repo_accessible(datastore_url, repo_id, max_retries=5, retry_delay=2):
    """Verify dataset repo is accessible (HTTP 200 on revision/main)."""
    url = f"{datastore_url}/v1/hf/api/datasets/{repo_id}/revision/main"
    for attempt in range(max_retries):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                print(f"✅ Verified: Dataset repository is accessible")
                return True
            if r.status_code == 500 and attempt < max_retries - 1:
                time.sleep(retry_delay)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                print(f"⚠️  Verification error: {e}")
    return False

def check_customizer_health(customizer_url):
    """Return True if Customizer /v1/health/ready returns 200."""
    try:
        r = requests.get(f"{customizer_url}/v1/health/ready", timeout=10)
        if r.status_code == 200:
            print("✅ Customizer service is healthy!")
            print(f"Response: {r.json()}")
            return True
        print(f"⚠️ Unexpected status: {r.status_code} - {r.text}")
    except requests.exceptions.RequestException as e:
        print(f"❌ Error connecting to customizer: {e}")
    return False

def check_datastore_health(datastore_url):
    """Return True if DataStore /v1/health returns 200."""
    try:
        r = requests.get(f"{datastore_url}/v1/health", timeout=10)
        if r.status_code == 200:
            d = r.json()
            print("✅ DataStore service is healthy!")
            print(f"Status: {d.get('status')}")
            return True
        print(f"⚠️ Unexpected status: {r.status_code} - {r.text}")
    except requests.exceptions.RequestException as e:
        print(f"❌ Error connecting to DataStore: {e}")
    return False

def check_entity_store_health(entity_store_url):
    """Return True if Entity Store /health returns 200."""
    try:
        r = requests.get(f"{entity_store_url}/health", timeout=10)
        if r.status_code == 200:
            print("✅ Entity Store service is healthy!")
            print(f"Status: {r.json().get('status')}")
            return True
        print(f"⚠️ Unexpected status: {r.status_code} - {r.text}")
    except requests.exceptions.RequestException as e:
        print(f"❌ Error connecting to Entity Store: {e}")
    return False

def ensure_fresh_dataset_repo(hf_api, repo_id, datastore_url):
    """Delete repo if it exists, then create it (for clean structure). Waits for git init."""
    try:
        r = requests.get(f"{datastore_url}/v1/hf/api/datasets/{repo_id}/revision/main", timeout=10)
        if r.status_code == 200:
            print(f"⚠️  Repo exists - deleting to recreate with correct structure")
            try:
                hf_api.delete_repo(repo_id=repo_id, repo_type="dataset")
                print(f"   ✅ Deleted")
            except Exception as e:
                print(f"   ℹ️  Delete: {e}")
    except Exception:
        pass
    hf_api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
    print(f"✅ Created dataset repo: {repo_id}")
    print(f"   ⏳ Waiting 3s for git repository initialization...")
    time.sleep(3)
    print(f"✅ Repository ready for uploads")

def wait_for_customization_job(job_id, customizer_url, polling_interval=30):
    """Poll job status until completed or failed. Returns final status or None."""
    start = time.time()
    print(f"Waiting for job {job_id}... (poll every {polling_interval}s)")
    try:
        r = requests.get(f"{customizer_url}/v1/customization/jobs/{job_id}/status", timeout=10)
        if r.status_code != 200:
            print(f"❌ Failed to get status: {r.status_code}")
            return None
        status_data = r.json()
        job_status = status_data.get("status")
        print(f"Initial status: {job_status}")
        while job_status in ("created", "pending", "running"):
            time.sleep(polling_interval)
            r = requests.get(f"{customizer_url}/v1/customization/jobs/{job_id}/status", timeout=10)
            if r.status_code == 200:
                status_data = r.json()
                job_status = status_data.get("status")
                elapsed = time.time() - start
                if "progress" in status_data:
                    print(f"   [{elapsed:.0f}s] {job_status} - {status_data.get('progress')}")
                else:
                    print(f"   [{elapsed:.0f}s] {job_status}")
        return job_status
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

## Step 3: Test Customizer Service Health

In [ ]:
# Test Customizer health (uses helper from Step 2b)
print(f"Checking health at: {CUSTOMIZER_URL}/v1/health/ready")
if not check_customizer_health(CUSTOMIZER_URL):
    print("Please verify: oc get pods -n", NMS_NAMESPACE, "| grep customizer")

## Step 4: List Customization Jobs (API Verification)

In [ ]:
# List existing customization jobs (this verifies the API is working)
try:
    jobs_url = f"{CUSTOMIZER_URL}/v1/customization/jobs"
    print(f"Listing jobs from: {jobs_url}")
    response = requests.get(jobs_url, timeout=10)
    print(f"Status Code: {response.status_code}")
    if response.status_code == 200:
        jobs = response.json()
        job_list = jobs.get('data', [])
        print(f"✅ API is working! Found {len(job_list)} customization job(s):")
        if job_list:
            for job in job_list[:5]:  # Show first 5 jobs
                print(f"  - Job ID: {job.get('id')}, Status: {job.get('status')}, Created: {job.get('created_at')}")
            if len(job_list) > 5:
                print(f"  ... and {len(job_list) - 5} more jobs")
        else:
            print("  No jobs found (this is normal for a fresh deployment)")
        print(f"\nTotal jobs: {jobs.get('pagination', {}).get('total_results', len(job_list))}")
    else:
        print(f"⚠️ Unexpected status code: {response.status_code}")
        print(f"Response: {response.text}")
except requests.exceptions.RequestException as e:
    print(f"❌ Error listing jobs: {e}")

## Step 5: Test DataStore Connection (Required for Customization)

In [ ]:
# Test DataStore health (uses helper from Step 2b)
print(f"Checking DataStore health at: {DATASTORE_URL}/v1/health")
if not check_datastore_health(DATASTORE_URL):
    print("Please verify: oc get pods -n", NMS_NAMESPACE, "| grep datastore")

## Step 6: Verify Before Proceeding

Verify the cells above worked well before proceeding.

## Step 7: Test Entity Store Connection (Required for Model Registration)

In [ ]:
# Test Entity Store health (uses helper from Step 2b)
print(f"Checking Entity Store health at: {ENTITY_STORE_URL}/health")
if not check_entity_store_health(ENTITY_STORE_URL):
    print("Please verify: oc get pods -n", NMS_NAMESPACE, "| grep entitystore")

## Step 8: Test DataStore Upload (Create Namespace)

In [ ]:
# Test DataStore namespace creation (required for dataset uploads)
try:
    namespace_url = f"{DATASTORE_URL}/v1/datastore/namespaces/{NMS_NAMESPACE}"
    print(f"Checking namespace at: {namespace_url}")
    response = requests.get(
        namespace_url, 
        headers={"Authorization": f"Bearer {NDS_TOKEN}"},
        timeout=10
    )
    
    if response.status_code == 404:
        # Create namespace (Data Store API expects 'namespace' key, not 'name')
        print(f"Creating namespace: {NMS_NAMESPACE}")
        response = requests.post(
            f"{DATASTORE_URL}/v1/datastore/namespaces",
            json={"namespace": NMS_NAMESPACE},
            headers={"Authorization": f"Bearer {NDS_TOKEN}"},
            timeout=10
        )
        if response.status_code in (200, 201):
            print(f"✅ Created namespace: {NMS_NAMESPACE}")
        else:
            print(f"⚠️ Failed to create namespace: {response.status_code} - {response.text}")
    elif response.status_code == 200:
        print(f"✅ Namespace exists: {NMS_NAMESPACE}")
    else:
        print(f"⚠️ Unexpected status code: {response.status_code} - {response.text}")
except requests.exceptions.RequestException as e:
    print(f"❌ Error with DataStore namespace: {e}")

## Step 9: Test DataStore Upload (Sample Dataset)

In [ ]:
# Step 9a: Initialize and create dataset repo (uses helpers from Step 2b)
import json
import tempfile

TEST_DATASET_NAME = "customizer-test-dataset"
repo_id = f"{NMS_NAMESPACE}/{TEST_DATASET_NAME}"
ENTITY_STORE_DATASET_ID = None  # Will be set in Step 10

hf_api = get_hf_api(DATASTORE_URL, NDS_TOKEN)
print(f"Step 1: Initializing namespace in DataStore (if needed)...")
ensure_datastore_namespace(hf_api, NMS_NAMESPACE)
print(f"\nStep 2: Creating dataset repo in DataStore: {repo_id}")
create_or_reuse_dataset_repo(hf_api, repo_id, DATASTORE_URL)

In [ ]:
# Step 9b: Create sample training file and upload (uses helper from Step 2b)
print(f"\nStep 3: Creating sample training data...")
sample_data = {
    "messages": [
        {"role": "user", "content": "What is the capital of France?"},
        {"role": "assistant", "content": "The capital of France is Paris."}
    ]
}
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump(sample_data, f, indent=2)
    temp_file = f.name

print(f"\nStep 4: Uploading sample file to DataStore...")
upload_file_with_retry(hf_api, repo_id, temp_file, "training/sample.json", NMS_NAMESPACE, DATASTORE_URL)
# Verify file is in repo (avoids false success when "0 LFS files" or backend didn't persist)
try:
    files = hf_api.list_repo_files(repo_id=repo_id, repo_type="dataset")
    if files and "training/sample.json" in files:
        print(f"✅ Uploaded sample file to DataStore (verified: training/sample.json in repo)")
    else:
        print(f"⚠️ Upload completed but file not found in repo listing (got {len(files or [])} files).")
        print(f"   If you see '0 LFS files' above, small files are stored as regular (non-LFS); check Step 5 verification.")
    print(f"   Dataset location: hf://datasets/{repo_id}")
except Exception as e:
    print(f"⚠️ Could not verify upload: {e}")
    print(f"   Dataset location: hf://datasets/{repo_id}")

In [ ]:
# Step 9c: Verify upload and cleanup (uses helper from Step 2b)
print(f"\nStep 5: Verifying upload completed...")
verify_repo_accessible(DATASTORE_URL, repo_id)
if os.path.exists(temp_file):
    os.unlink(temp_file)

print(f"\n✅ Dataset upload test successful!")
print(f"   Dataset can be accessed via: hf://datasets/{repo_id}")

## Step 10: Test Entity Store Dataset Registration

In [ ]:
# Test registering the dataset in Entity Store
# IMPORTANT: Customizer needs datasets registered in Entity Store (not just DataStore)
# The Entity Store provides the dataset ID that Customizer uses

try:
    files_url = f"hf://datasets/{repo_id}"
    print(f"Registering dataset in Entity Store...")
    print(f"   Dataset name: {TEST_DATASET_NAME}")
    print(f"   Files URL: {files_url}")
    
    response = requests.post(
        f"{ENTITY_STORE_URL}/v1/datasets",
        json={
            "name": TEST_DATASET_NAME,
            "namespace": NMS_NAMESPACE,
            "description": "Test dataset for Customizer service verification",
            "files_url": files_url,
            "project": "customizer-test",
            "format": "json",
        },
        timeout=30
    )
    
    if response.status_code in (200, 201):
        dataset_obj = response.json()
        print(f"✅ Dataset registered in Entity Store!")
        print(f"   Dataset ID: {dataset_obj.get('id', 'N/A')}")
        print(f"   Files URL: {dataset_obj.get('files_url', 'N/A')}")
        ENTITY_STORE_DATASET_ID = dataset_obj.get('id')
    elif response.status_code == 409:
        print(f"ℹ️ Dataset {TEST_DATASET_NAME} already exists in Entity Store")
        # Try to get the existing dataset
        get_response = requests.get(
            f"{ENTITY_STORE_URL}/v1/datasets/{NMS_NAMESPACE}/{TEST_DATASET_NAME}",
            timeout=30
        )
        if get_response.status_code == 200:
            dataset_obj = get_response.json()
            ENTITY_STORE_DATASET_ID = dataset_obj.get('id')
            print(f"   Dataset ID: {ENTITY_STORE_DATASET_ID}")
            print(f"   Files URL: {dataset_obj.get('files_url', 'N/A')}")
    else:
        print(f"⚠️ Failed to register in Entity Store: {response.status_code}")
        print(f"   Response: {response.text}")
        ENTITY_STORE_DATASET_ID = None
        
except requests.exceptions.RequestException as e:
    print(f"❌ Error registering dataset in Entity Store: {e}")
    ENTITY_STORE_DATASET_ID = None

## Step 11: Verify Customizer Can Find the Dataset

In [ ]:
# Verify that Customizer can access the dataset via Entity Store
# Customizer uses Entity Store to resolve dataset IDs to actual file locations

if ENTITY_STORE_DATASET_ID:
    print(f"Verifying Customizer can access dataset via Entity Store...")
    print(f"   Entity Store Dataset ID: {ENTITY_STORE_DATASET_ID}")
    
    # Check if Entity Store has the dataset
    try:
        get_response = requests.get(
            f"{ENTITY_STORE_URL}/v1/datasets/{NMS_NAMESPACE}/{TEST_DATASET_NAME}",
            timeout=30
        )
        if get_response.status_code == 200:
            dataset_info = get_response.json()
            print(f"✅ Entity Store has the dataset")
            print(f"   Name: {dataset_info.get('name')}")
            print(f"   Files URL: {dataset_info.get('files_url')}")
            print(f"   Format: {dataset_info.get('format')}")
            
            # Verify the files_url points to DataStore
            files_url = dataset_info.get('files_url', '')
            if files_url.startswith('hf://datasets/'):
                print(f"✅ Dataset files URL is correctly formatted for DataStore")
                print(f"   Customizer can use this dataset ID: {ENTITY_STORE_DATASET_ID}")
            else:
                print(f"⚠️ Unexpected files URL format: {files_url}")
        else:
            print(f"⚠️ Could not retrieve dataset from Entity Store: {get_response.status_code}")
    except Exception as e:
        print(f"❌ Error verifying dataset: {e}")
else:
    print("⚠️ Skipping verification - dataset not registered in Entity Store")

## Step 12: Test Base Model (Before Customization)

Before customizing, let's test the base model to see its responses. This will help us verify that customization actually changed the model's behavior.

In [ ]:
# Test the base model with sample prompts (before customization)
# This helps verify that customization actually changed the model's behavior

if not INFERENCE_SERVICE_URL:
    print("⚠️  INFERENCE_SERVICE_URL not set in env.donotcommit")
    print("   Skipping base model test - set INFERENCE_SERVICE_URL to test model responses")
    print("   Example: INFERENCE_SERVICE_URL=http://your-inference-service:8000")
    base_responses = {}
else:
    # Disable SSL warnings for HTTPS
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    # Headers: try without Bearer first for internal URL (in-cluster often bypasses OAuth)
    headers_with_auth = {"Content-Type": "application/json"}
    if NIM_SERVICE_ACCOUNT_TOKEN:
        headers_with_auth["Authorization"] = f"Bearer {NIM_SERVICE_ACCOUNT_TOKEN}"
    headers_no_auth = {"Content-Type": "application/json"}
    
    # Build URLs: internal HTTP:80 first (bypasses OAuth), then external. No duplicate entries.
    potential_urls = []
    if INFERENCE_SERVICE_NAME:
        internal_http = f"http://{INFERENCE_SERVICE_NAME}-predictor.{NMS_NAMESPACE}.svc.cluster.local:80"
        potential_urls.append((internal_http, True))   # try without auth first
    potential_urls.append((INFERENCE_SERVICE_URL, False))
    seen = set()
    unique_urls = [(u, no_auth) for u, no_auth in potential_urls if u not in seen and not seen.add(u)]
    
    test_model_name = INFERENCE_SERVICE_NAME if INFERENCE_SERVICE_NAME else (BASE_MODEL if BASE_MODEL else None)
    working_url = None
    print("🔍 Detecting working InferenceService URL...")
    for test_url, try_no_auth_first in unique_urls:
        if not test_model_name:
            continue
        test_payload = {"model": test_model_name, "messages": [{"role": "user", "content": "test"}], "max_tokens": 5}
        headers_list = [headers_no_auth, headers_with_auth] if (try_no_auth_first and NIM_SERVICE_ACCOUNT_TOKEN) else [headers_with_auth]
        for headers in headers_list:
            try:
                response = requests.post(
                    f"{test_url}/v1/chat/completions",
                    headers=headers,
                    json=test_payload,
                    timeout=10,
                    verify=False
                )
                if response.status_code == 200:
                    working_url = test_url
                    print(f"✅ Found working URL: {test_url}")
                    break
                elif response.status_code == 503:
                    print(f"⚠️  {test_url} returned 503 (service unavailable), trying next...")
                elif response.status_code == 400:
                    err = (response.text or "")[:220]
                    print(f"⚠️  {test_url} returned 400 (Bad Request): {err}")
                    if "authorization server" in err.lower() or "server_error" in err:
                        print("   💡 Hint: URL may be behind OAuth. From Workbench use internal HTTP: " + (f"http://{INFERENCE_SERVICE_NAME}-predictor.{NMS_NAMESPACE}.svc.cluster.local:80" if INFERENCE_SERVICE_NAME else ""))
                else:
                    print(f"⚠️  {test_url} returned {response.status_code}: {(response.text or '')[:200]}")
            except Exception as e:
                print(f"⚠️  {test_url} failed: {str(e)[:60]}, trying next...")
            if working_url:
                break
        if working_url:
            break
    
    if not working_url:
        print("❌ No working URL found. Cannot test base model.")
        print("   Please check:")
        print(f"   1. InferenceService is running: oc get inferenceservice {INFERENCE_SERVICE_NAME} -n {NMS_NAMESPACE}")
        print(f"   2. Pods are ready: oc get pods -n {NMS_NAMESPACE} | grep {INFERENCE_SERVICE_NAME}")
        print(f"   3. From Workbench (in-cluster), set internal HTTP in env: INFERENCE_SERVICE_URL=http://{INFERENCE_SERVICE_NAME}-predictor.{NMS_NAMESPACE}.svc.cluster.local:80")
        print(f"   4. External URL may be behind OAuth; service account token may not apply there.")
        base_responses = {}
    else:
        # Sample prompts to test (continued in next cell)
        pass


In [ ]:
test_prompts = [
    "What personal data does Red Hat collect?",
    "How does Red Hat use my personal data?",
    "What are my privacy rights with Red Hat?"
]

# Store base model responses
base_responses = {}
try:
    _url = working_url
except NameError:
    _url = None
if not _url:
    print("⚠️ No working URL from previous cell - skipping base model inference.")
    print("   Run the previous cell first and ensure a working InferenceService URL is found.")
else:
    print(f"\nTesting base model responses using: {working_url}")
    print("=" * 60)

    # vLLM uses the InferenceService name as the served model name (--served-model-name)
    # So we use INFERENCE_SERVICE_NAME as the model name in API requests
    # Fallback to BASE_MODEL if INFERENCE_SERVICE_NAME is not set
    model_names_to_try = []
    if INFERENCE_SERVICE_NAME:
        model_names_to_try.append(INFERENCE_SERVICE_NAME)  # Primary: use InferenceService name (e.g., your-inferenceservice-name)
    if BASE_MODEL:
        model_names_to_try.append(BASE_MODEL)  # Fallback: try BASE_MODEL (e.g., meta/llama-3.2-1b-instruct)
        # Also try variations of BASE_MODEL as additional fallbacks
        if "meta/" in BASE_MODEL:
            model_names_to_try.append(BASE_MODEL.replace("meta/", ""))  # Without prefix
            model_names_to_try.append(f"models/{BASE_MODEL.replace('meta/', '')}")  # With models/ prefix
    # Remove duplicates while preserving order
    model_names_to_try = list(dict.fromkeys(model_names_to_try))

    if not model_names_to_try:
        print("⚠️  No model name available - set INFERENCE_SERVICE_NAME or BASE_MODEL in env.donotcommit")
        base_responses = {}
    else:
        for prompt in test_prompts:
            print(f"\n📝 Prompt: {prompt}")
        
            try:
                # Use OpenAI-compatible API endpoint
                success = False
                for model_name in model_names_to_try:
                    if success:
                        break
                    
                    payload = {
                        "model": model_name,
                        "messages": [
                            {"role": "user", "content": prompt}
                        ],
                        "temperature": 0.7,
                        "max_tokens": 150
                    }
                
                    try:
                        response = requests.post(
                            f"{working_url}/v1/chat/completions",
                            headers=headers_with_auth,
                            json=payload,
                            timeout=30,
                            verify=False  # Disable SSL verification for HTTPS
                        )
                    
                        if response.status_code == 200:
                            result = response.json()
                            answer = result["choices"][0]["message"]["content"]
                            base_responses[prompt] = answer
                            print(f"✅ Success with model '{model_name}'!")
                            print(f"✅ Response: {answer[:100]}...")
                            success = True
                            break  # Success, no need to try other model names
                        elif response.status_code == 404:
                            # Model not found - try next model name format
                            error_msg = response.text[:200]
                            if "does not exist" in error_msg or "not found" in error_msg.lower():
                                print(f"   ⚠️  Model '{model_name}' not found, trying next format...")
                                break  # Try next model name
                            else:
                                print(f"   ❌ Error 404: {error_msg}")
                                if model_name == model_names_to_try[-1]:  # Last attempt
                                    base_responses[prompt] = None
                                continue
                        else:
                            print(f"   ❌ Error {response.status_code}: {response.text[:200]}")
                            if model_name == model_names_to_try[-1]:  # Last attempt
                                base_responses[prompt] = None
                            continue  # Try next model name
                    except requests.exceptions.ConnectionError as ce:
                        print(f"   ❌ Connection error: {ce}")
                        if model_name == model_names_to_try[-1]:  # Last attempt
                            base_responses[prompt] = None
                        continue  # Try next model name
                    except Exception as e:
                        print(f"   ❌ Exception: {e}")
                        if model_name == model_names_to_try[-1]:  # Last attempt
                            base_responses[prompt] = None
                        continue  # Try next model name
            
                if not success:
                    base_responses[prompt] = None
                    print(f"   ❌ Failed with all model name formats: {model_names_to_try}")
                
            except Exception as e:
                print(f"❌ Exception: {e}")
                base_responses[prompt] = None
    
        print("\n" + "=" * 60)
        print(f"✅ Base model testing complete. Saved {len([r for r in base_responses.values() if r])} responses.")
        # Save full responses to file (Jupyter shortens with "..." on UI)
        base_responses_file = Path("base_model_responses.json")
        base_responses_txt = Path("base_model_responses.txt")
        with open(base_responses_file, "w", encoding="utf-8") as f:
            json.dump(base_responses, f, indent=2, ensure_ascii=False)
        with open(base_responses_txt, "w", encoding="utf-8") as f:
            for prompt, answer in base_responses.items():
                f.write(f"Prompt: {prompt}\n")
                f.write(f"Response: {answer if answer is not None else '(no response)'}\n")
                f.write("-" * 60 + "\n")
        print(f"📁 Full responses saved to: {base_responses_file.absolute()} and {base_responses_txt.absolute()}")

    print("\n" + "=" * 60)
    print(f"✅ Base model testing complete. Saved {len([r for r in base_responses.values() if r])} responses.")



## Step 13: Prepare Training Data for Customization

Now let's prepare training data in the correct format for customization. We'll use OpenAI format (messages) as required by NeMo Customizer.

In [ ]:
# Prepare training data for customization
# IMPORTANT: Must use OpenAI format (messages) as per NVIDIA reference
# IMPORTANT: Customizer requires BOTH training AND validation files
# Reference: https://github.com/NVIDIA/k8s-nim-operator/blob/main/test/e2e/jupyter-notebook/e2e-notebook.ipynb

# Use TEST_DATASET_NAME from Step 9 if available, otherwise use default
if 'TEST_DATASET_NAME' not in locals():
    TEST_DATASET_NAME = "customizer-test-dataset"
CUSTOMIZATION_DATASET_NAME = f"{TEST_DATASET_NAME}-customization"

# Load training data from external file
# This makes it easier to add more examples without modifying the notebook
import json
from pathlib import Path

training_dataset_path = Path("training_dataset.json")
if not training_dataset_path.exists():
    # Try in the same directory as the notebook (if running in Jupyter)
    training_dataset_path = Path.cwd() / "training_dataset.json"

if training_dataset_path.exists():
    print(f"📂 Loading training data from: {training_dataset_path}")
    with open(training_dataset_path, 'r', encoding='utf-8') as f:
        all_training_data = json.load(f)
    print(f"✅ Loaded {len(all_training_data)} examples from training_dataset.json")
else:
    print(f"⚠️  training_dataset.json not found at {training_dataset_path}")
    print("   Using minimal fallback dataset (you should create training_dataset.json with more examples)")
    print("   See training_dataset.json.example for format")
    # Minimal fallback dataset (6 examples - Red Hat privacy policy)
    all_training_data = [
    {
        "messages": [
            {"role": "user", "content": "What personal data does Red Hat collect?"},
            {"role": "assistant", "content": "Red Hat collects contact information (name, email, phone, address), account and commercial information (account number, username, payment info, purchase records), professional information (job applications), internet activity (IP address, browsing history, browser type), location information, and inferences drawn from your data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How does Red Hat collect my personal data?"},
            {"role": "assistant", "content": "Red Hat collects personal data directly from you when you create an account, make purchases, request support, register for events, participate in surveys, or apply for employment. Red Hat also collects data indirectly through website interactions, cookies, and from third parties like your employer or business partners."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What does Red Hat use my personal data for?"},
            {"role": "assistant", "content": "Red Hat uses your personal data to verify your identity, fulfill requests for products and services, provide support, personalize your online experience, improve websites and services, conduct market research, and comply with legal obligations."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Does Red Hat share my personal data with third parties?"},
            {"role": "assistant", "content": "Red Hat may share personal data with business partners and service providers (distributors, resellers, payment processors), IBM and affiliates, as required by law or legal process, and in business transactions like mergers or acquisitions. Partners are required by contract to keep information confidential and secure."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What are my privacy rights with Red Hat?"},
            {"role": "assistant", "content": "You have rights to access, rectify, erase, restrict use, object to processing, request data portability, withdraw consent, and lodge complaints. Submit requests via the Personal Data Request Form at redhat.com/en/about/personal-data-request or contact privacy@redhat.com."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What types of cookies does Red Hat use?"},
            {"role": "assistant", "content": "Red Hat uses required cookies (necessary for website functionality), functional cookies (for analytics and personalization), and advertising cookies (for targeted ads). You can disable functional and advertising cookies through browser settings or Red Hat's cookie manager."}
        ]
    }
]

# Split into training (80%) and validation (20%) sets
# IMPORTANT: Validation set must have at least as many samples as batch_size (8)
# With 50 examples and 80/20 split: 40 train, 10 val (validation has enough for batch_size 8)
import math
split_idx = math.floor(len(all_training_data) * 0.8)  # Use floor to ensure validation gets enough
training_data = all_training_data[:split_idx]
validation_data = all_training_data[split_idx:]

# Ensure validation has at least 8 samples (matching batch_size)
if len(validation_data) < 8:
    # Move samples from training to validation if needed
    while len(validation_data) < 8 and len(training_data) > 0:
        validation_data.append(training_data.pop())

print(f"✅ Prepared {len(all_training_data)} total examples")
print(f"   Training examples: {len(training_data)}")
print(f"   Validation examples: {len(validation_data)}")
print("\nSample training example:")
print(json.dumps(training_data[0], indent=2))
print("\nSample validation example:")
print(json.dumps(validation_data[0], indent=2))

## Step 14: Upload Training Data to DataStore

Upload the training data to DataStore using the HuggingFace-compatible API.

In [ ]:
# Step 14a: Upload training data to DataStore (uses helpers from Step 2b)
import tempfile

customization_repo_id = f"{NMS_NAMESPACE}/{CUSTOMIZATION_DATASET_NAME}"

hf_api = get_hf_api(DATASTORE_URL, NDS_TOKEN)
print(f"Step 1: Initializing namespace in DataStore (if needed)...")
ensure_datastore_namespace(hf_api, NMS_NAMESPACE)
print(f"\nStep 2: Creating/updating dataset repo in DataStore: {customization_repo_id}")
ensure_fresh_dataset_repo(hf_api, customization_repo_id, DATASTORE_URL)


In [ ]:
print(f"\nStep 3: Creating training and validation data files...")

# Create training file
with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False) as f:
    for item in training_data:
        f.write(json.dumps(item) + '\n')
    temp_train_file = f.name

# Create validation file
with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False) as f:
    for item in validation_data:
        f.write(json.dumps(item) + '\n')
    temp_val_file = f.name

try:
    print(f"\nStep 4: Uploading training and validation data to DataStore...")
    upload_succeeded = False
    max_upload_retries = 2
    
    for upload_attempt in range(max_upload_retries):
        try:
            # NeMo Customizer 25.x (25.08, 25.12+) requires: training/train.jsonl and validation/validation.jsonl
            # Tested locally - confirmed this structure works for all 25.x versions
            hf_api.upload_file(
                path_or_fileobj=temp_train_file,
                path_in_repo="training/train.jsonl",
                repo_id=customization_repo_id,
                repo_type="dataset"
            )
            print(f"✅ Uploaded: training/train.jsonl ({len(training_data)} samples)")
            
            hf_api.upload_file(
                path_or_fileobj=temp_val_file,
                path_in_repo="validation/validation.jsonl",
                repo_id=customization_repo_id,
                repo_type="dataset"
            )
            print(f"✅ Uploaded: validation/validation.jsonl ({len(validation_data)} samples)")
            
            print(f"\n✅ Uploaded both training and validation data to DataStore")
            print(f"   Dataset location: hf://datasets/{customization_repo_id}")
            print(f"   Training samples: {len(training_data)}")
            print(f"   Validation samples: {len(validation_data)}")
            upload_succeeded = True
            break
        except Exception as upload_error:
            error_str = str(upload_error).lower()
            # Check if it's a 500 error (corrupted repo) and we can retry
            if ("500" in error_str or "internal server error" in error_str) and upload_attempt < max_upload_retries - 1:
                print(f"⚠️  Upload failed with 500 error (attempt {upload_attempt + 1}/{max_upload_retries})")
                print(f"   Repository may have become corrupted - will delete and recreate...")
                try:
                    hf_api.delete_repo(repo_id=customization_repo_id, repo_type="dataset")
                    print(f"   ✅ Deleted corrupted repo")
                except:
                    pass
                
                # Recreate with namespace init
                temp_repo_id = f"{NMS_NAMESPACE}/namespace-init-temp"
                try:
                    hf_api.create_repo(repo_id=temp_repo_id, repo_type="dataset", exist_ok=True)
                    hf_api.delete_repo(repo_id=temp_repo_id, repo_type="dataset")
                except:
                    pass
                
                hf_api.create_repo(repo_id=customization_repo_id, repo_type="dataset", exist_ok=True)
                print(f"   ✅ Recreated repo")
                import time
                time.sleep(3)
                print(f"   ✅ Waited for git init - retrying upload...")
            else:
                # Not a retryable error or last attempt
                raise
    
    if not upload_succeeded:
        raise Exception("Upload failed after retries")
    
    # Step 5: Verify upload completed by listing repository files
    # This ensures the commit succeeded and repository is accessible
    print(f"\nStep 5: Verifying upload completed...")
    import time
    max_retries = 5
    retry_delay = 2
    
    for attempt in range(max_retries):
        try:
            repo_files = hf_api.list_repo_files(repo_id=customization_repo_id, repo_type="dataset")
            if repo_files:
                print(f"✅ Verified: Repository contains {len(repo_files)} file(s)")
                for file in repo_files:
                    print(f"   - {file}")
                break
            else:
                if attempt < max_retries - 1:
                    print(f"   ⏳ Repository empty, waiting {retry_delay}s for commit to complete... (attempt {attempt + 1}/{max_retries})")
                    time.sleep(retry_delay)
                else:
                    raise Exception("Repository is empty after upload - commit may have failed!")
        except Exception as verify_error:
            if attempt < max_retries - 1:
                print(f"   ⏳ Verification failed, retrying in {retry_delay}s... (attempt {attempt + 1}/{max_retries})")
                print(f"   Error: {str(verify_error)[:100]}")
                time.sleep(retry_delay)
            else:
                print(f"❌ Upload verification failed after {max_retries} attempts: {verify_error}")
                print(f"   The customization job will fail if the dataset is not accessible!")
                raise
finally:
    # Clean up temp files
    import os
    if 'temp_train_file' in locals() and os.path.exists(temp_train_file):
        os.unlink(temp_train_file)
    if 'temp_val_file' in locals() and os.path.exists(temp_val_file):
        os.unlink(temp_val_file)

print(f"\n✅ Training data upload and verification successful!")



In [ ]:
# Helper: Generate unique output model name with retry logic for job creation
# This helps avoid "model already exists" errors from DataStore

import random

def create_customization_job_with_retry(customizer_url, training_params_template, max_retries=3):
    """
    Create a customization job with automatic retry if model repo already exists.
    
    Args:
        customizer_url: Base URL for Customizer service
        training_params_template: Template dict for training params (will be modified)
        max_retries: Maximum number of retry attempts
    
    Returns:
        tuple: (job_id, job_response) or (None, None) if failed
    """
    headers_custom = {"Content-Type": "application/json"}
    
    for attempt in range(max_retries):
        # Generate unique timestamp with random offset
        timestamp = int(time.time()) + random.randint(1, 10000)
        
        # IMPORTANT: Use NMS_NAMESPACE as the namespace (not "meta") to avoid conflicts
        # Extract model name from BASE_MODEL (e.g., "meta/llama-3.2-1b-instruct" -> "llama-3.2-1b-instruct")
        if "/" in BASE_MODEL:
            model_name_only = BASE_MODEL.split("/", 1)[1]  # Get part after namespace
        else:
            model_name_only = BASE_MODEL
        
        # Use NMS_NAMESPACE as namespace and add "-custom" suffix to distinguish from base model
        unique_output_model = f"{NMS_NAMESPACE}/{model_name_only}-custom@1.0-{timestamp}"
        
        # Update training params with new model name
        training_params = training_params_template.copy()
        training_params["output_model"] = unique_output_model
        training_params["name"] = f"customizer-test-job-{timestamp}"
        
        if attempt > 0:
            print(f"\n🔄 Retry {attempt + 1}/{max_retries}: Using output model: {unique_output_model}")
        else:
            print(f"\n📋 Using unique output model: {unique_output_model}")
        
        try:
            response = requests.post(
                f"{customizer_url}/v1/customization/jobs",
                json=training_params,
                headers=headers_custom,
                timeout=60
            )
            
            if response.status_code in (200, 201):
                job_response = response.json()
                job_id = job_response.get("id")
                return job_id, job_response
            elif response.status_code == 500:
                error_text = response.text.lower()
                if ("already exists" in error_text or "model or dataset" in error_text) and attempt < max_retries - 1:
                    print(f"⚠️  Model repo already exists, retrying with different timestamp...")
                    time.sleep(1)
                    continue
                else:
                    print(f"❌ Error: {response.text[:500]}")
                    return None, None
            else:
                print(f"❌ HTTP {response.status_code}: {response.text[:500]}")
                return None, None
                
        except Exception as e:
            print(f"❌ Exception: {e}")
            if attempt < max_retries - 1:
                time.sleep(1)
                continue
            return None, None
    
    return None, None

print("✅ Helper function loaded: create_customization_job_with_retry()")

## Step 15: Register Customization Dataset in Entity Store

Register the customization dataset in Entity Store so the Customizer can use it.

In [ ]:
# Register customization dataset in Entity Store
# IMPORTANT: Verify dataset exists in DataStore before registering in Entity Store
# Customizer validates datasets by calling DataStore API, so the repository must exist

try:
    files_url = f"hf://datasets/{customization_repo_id}"
    print(f"Step 1: Verifying dataset exists in DataStore...")
    print(f"   Dataset: {customization_repo_id}")
    
    # Verify dataset repository exists in DataStore before Entity Store registration
    # Customizer will call DataStore API to validate, so this must succeed
    import time
    max_verify_retries = 10
    verify_retry_delay = 3
    dataset_verified = False
    
    for verify_attempt in range(max_verify_retries):
        try:
            # Try to access the dataset revision endpoint (what Customizer checks)
            verify_url = f"{DATASTORE_URL}/v1/hf/api/datasets/{customization_repo_id}/revision/main"
            verify_response = requests.get(verify_url, timeout=10)
            
            if verify_response.status_code == 200:
                print(f"✅ Dataset verified in DataStore!")
                dataset_verified = True
                break
            elif verify_response.status_code == 500:
                if verify_attempt < max_verify_retries - 1:
                    print(f"   ⏳ Dataset repository not ready yet, waiting {verify_retry_delay}s... (attempt {verify_attempt + 1}/{max_verify_retries})")
                    print(f"   Error: Repository may not exist - commit may still be in progress")
                    time.sleep(verify_retry_delay)
                else:
                    print(f"❌ Dataset repository does not exist in DataStore after {max_verify_retries} attempts")
                    print(f"   This means the upload commit failed. Please re-run Step 14 (upload) cell.")
                    raise Exception(f"Dataset {customization_repo_id} not found in DataStore - upload may have failed")
            else:
                print(f"⚠️ Unexpected status code from DataStore: {verify_response.status_code}")
                print(f"   Response: {verify_response.text[:200]}")
                if verify_attempt < max_verify_retries - 1:
                    time.sleep(verify_retry_delay)
                else:
                    raise Exception(f"Could not verify dataset in DataStore: {verify_response.status_code}")
        except requests.exceptions.RequestException as verify_error:
            if verify_attempt < max_verify_retries - 1:
                print(f"   ⏳ Verification request failed, retrying in {verify_retry_delay}s... (attempt {verify_attempt + 1}/{max_verify_retries})")
                print(f"   Error: {str(verify_error)[:100]}")
                time.sleep(verify_retry_delay)
            else:
                raise Exception(f"Failed to verify dataset in DataStore: {verify_error}")
    
    if not dataset_verified:
        raise Exception("Dataset verification failed - cannot proceed with Entity Store registration")
    
    print(f"\nStep 2: Registering customization dataset in Entity Store...")
    print(f"   Dataset name: {CUSTOMIZATION_DATASET_NAME}")
    print(f"   Files URL: {files_url}")
    
    response = requests.post(
        f"{ENTITY_STORE_URL}/v1/datasets",
        json={
            "name": CUSTOMIZATION_DATASET_NAME,
            "namespace": NMS_NAMESPACE,
            "description": "Training data for model customization",
            "files_url": files_url,
            "project": "customizer-test",
            "format": "json",
        },
        timeout=30
    )
    
    if response.status_code in (200, 201):
        dataset_obj = response.json()
        print(f"✅ Customization dataset registered in Entity Store!")
        print(f"   Dataset ID: {dataset_obj.get('id', 'N/A')}")
        print(f"   Files URL: {dataset_obj.get('files_url', 'N/A')}")
        customization_dataset_registered = True
    elif response.status_code == 409:
        print(f"ℹ️ Dataset {CUSTOMIZATION_DATASET_NAME} already exists in Entity Store")
        customization_dataset_registered = True
    else:
        print(f"⚠️ Failed to register in Entity Store: {response.status_code}")
        print(f"   Response: {response.text}")
        customization_dataset_registered = False
        
except requests.exceptions.RequestException as e:
    print(f"❌ Error registering dataset in Entity Store: {e}")
    customization_dataset_registered = False
except Exception as e:
    print(f"❌ Error: {e}")
    print(f"\n💡 Troubleshooting:")
    print(f"   1. Re-run Step 14 (Upload Training Data to DataStore) cell")
    print(f"   2. Wait for the verification step to complete successfully")
    print(f"   3. Then re-run this cell (Step 15)")
    customization_dataset_registered = False

## Step 16: Create Customization Job

Create a customization job using the Customizer API to fine-tune the base model with our training data.

**⚠️ GPU Requirements:**
- Each training job requires 1 GPU
- Ensure GPUs are available before creating the job
- If all GPUs are busy, training pods will remain in `Pending` state
- To check GPU availability: `oc get nodes -o json | grep nvidia.com/gpu`
- To free up GPUs, scale down non-essential services: `oc scale deployment <service-name> -n <namespace> --replicas=0`

In [ ]:
# Create customization job using Customizer API
# This follows the official NVIDIA k8s-nim-operator e2e notebook implementation

if not customization_dataset_registered:
    print("❌ Cannot create customization job - dataset not registered")
    job_id = None
    job_response = None
else:
    print(f"✅ Prerequisites validated")
    print(f"  Base Model Config: {CUSTOMIZATION_TEMPLATE}")
    print(f"  Dataset Name: {CUSTOMIZATION_DATASET_NAME}")
    print(f"  Dataset Namespace: {NMS_NAMESPACE}")
    print(f"  Dataset URL: hf://datasets/{customization_repo_id}")
    
    # Generate unique output model name with timestamp to avoid conflicts
    # IMPORTANT: Use retry logic to handle "model already exists" errors
    import random
    max_retries = 3
    job_created = False
    job_id = None
    job_response = None
    


In [ ]:
for attempt in range(max_retries):
        timestamp = int(time.time())
        random_suffix = random.randint(10000, 99999)  # Add random component for uniqueness

        # Extract model name from BASE_MODEL (e.g., "meta/llama-3.2-1b-instruct" -> "llama-3.2-1b-instruct")
        if "/" in BASE_MODEL:
            model_name_only = BASE_MODEL.split("/", 1)[1]  # Get part after namespace
        else:
            model_name_only = BASE_MODEL

        # IMPORTANT: Include timestamp in model name (not just version) to avoid conflicts
        # Customizer checks if model name exists, so we need unique model names, not just unique versions
        # Format: namespace/model-name-custom-timestamp-random@version
        unique_output_model = f"{NMS_NAMESPACE}/{model_name_only}-custom-{timestamp}-{random_suffix}@1.0"

        if attempt > 0:
            print(f"\n🔄 Retry {attempt + 1}/{max_retries}: Using output model: {unique_output_model}")
        else:
            print(f"📋 Using unique output model: {unique_output_model}")
            print(f"   Namespace: {NMS_NAMESPACE} (avoids conflicts with 'meta' namespace)")
            print(f"   Timestamp + Random: {timestamp}-{random_suffix} (ensures uniqueness)")

        # IMPORTANT: Payload format based on tested API requirements
        # - config: Must be a STRING (model version), not a dict!
        # - dataset: Must be a STRING (hf://datasets/...), not a dict!
        # - hyperparameters: Must be a dict with all required fields
        training_params = {
            "base_model": BASE_MODEL,
            "dataset": f"hf://datasets/{customization_repo_id}",  # String, not dict!
            "output_model": unique_output_model,
            "template": CUSTOMIZATION_TEMPLATE,
            "config": f"{BASE_MODEL}@v1.0.0",  # String format: model@version
                            "hyperparameters": {
                    "finetuning_type": "lora",
                    "training_type": "sft",
                    "batch_size": 8,  # Keep at 8 - validation set will have enough samples
                    "epochs": 1,
                    "learning_rate": 0.0001,
                "lora": {
                    "adapter_dim": 16,
                    "alpha": 16,
                    "adapter_dropout": 0.1,
                    "target_modules": None
                },
                "sequence_packing_enabled": False
            }
        }

        if attempt == 0:
            print(f"\n📋 Request payload:")
            print(json.dumps(training_params, indent=2))

        try:
            if attempt == 0:
                print(f"\n🚀 Creating customization job...")
                print(f"   Customizer URL: {CUSTOMIZER_URL}")
                print(f"   Endpoint: /v1/customization/jobs")

            headers_custom = {"Content-Type": "application/json"}

            response = requests.post(
                f"{CUSTOMIZER_URL}/v1/customization/jobs",
                json=training_params,
                headers=headers_custom,
                timeout=60
            )

            if response.status_code in (200, 201):
                job_response = response.json()
                job_id = job_response.get("id")
                print(f"✅ Customization job created successfully!")
                print(f"   Job ID: {job_id}")
                print(f"   Output Model: {unique_output_model}")
                print(f"   Status: {job_response.get('status', 'N/A')}")
                print(f"\n💡 Next steps:")
                print(f"   1. The job will create a Volcano Job and training worker pod")
                print(f"   2. Training worker requires 1 GPU - ensure GPUs are available")
                print(f"   3. Check training pod: oc get pods -n {NMS_NAMESPACE} | grep {job_id}")
                print(f"   4. Check Volcano Job: oc get vj -n {NMS_NAMESPACE} | grep {job_id}")
                print(f"   5. If pod is Pending, check GPU availability or scale down other services")
                job_created = True
                break
            elif response.status_code == 500:
                error_text = response.text.lower()
                if "already exists" in error_text and attempt < max_retries - 1:
                    print(f"⚠️  Model repo already exists, retrying with different name...")
                    time.sleep(1)
                    continue
                else:
                    print(f"❌ Failed to create customization job (500 error)")
                    print(f"   Response: {response.text[:500]}")
                    if attempt == max_retries - 1:
                        job_id = None
                        job_response = None
            else:
                print(f"❌ Failed to create customization job")
                print(f"   HTTP Status: {response.status_code}")
                print(f"   Response: {response.text[:500]}")
                if attempt == max_retries - 1:
                    job_id = None
                    job_response = None
                    break

        except Exception as e:
            print(f"❌ Exception creating customization job: {e}")
            if attempt < max_retries - 1:
                print(f"   Retrying with different model name...")
                time.sleep(1)
                continue
            else:
                import traceback
                traceback.print_exc()
                job_id = None
                job_response = None

if not job_created:
        print(f"\n❌ Failed to create customization job after {max_retries} attempts")
        print(f"   Please check:")
        print(f"   1. Dataset exists in DataStore: {customization_repo_id}")
        print(f"   2. Dataset is registered in Entity Store")
        print(f"   3. Customizer service is running")
        job_id = None
        job_response = None



## Step 17: Wait for Customization Job to Complete

Monitor the customization job until it completes. This may take several minutes depending on the dataset size and model.

In [ ]:
# Wait for customization job (uses helper from Step 2b)
if 'job_id' in locals() and job_id:
    job_status = wait_for_customization_job(job_id, CUSTOMIZER_URL)
    if job_status:
        if job_status == 'completed':
            print(f"\n✅ Customization job completed successfully!")
            print(f"   The customized model should now be available")
        elif job_status == 'failed':
            print(f"\n❌ Customization job FAILED")
            print(f"   Check Customizer logs: oc logs -n {NMS_NAMESPACE} -l app=nemocustomizer --tail=200")
        else:
            print(f"\n⚠️  Customization job finished with status: {job_status}")
    else:
        print(f"\n⚠️  Could not determine job status")
else:
    print("⚠️  No job ID available - skipping wait step")
    print("   Make sure to run the 'Create Customization Job' cell first")
    job_status = None

In [ ]:
# Verify customization job completed successfully
if 'job_status' in locals() and job_status == 'completed':
    print("✅ Customization job completed successfully!")
    print(f"\n📦 Customized model details:")
    if 'unique_output_model' in locals():
        print(f"   Output model: {unique_output_model}")
        customized_model_name = unique_output_model
    elif 'job_response' in locals() and job_response:
        output_model = job_response.get('output_model', 'N/A')
        print(f"   Output model: {output_model}")
        customized_model_name = output_model
    else:
        customized_model_name = None
    
    print(f"   Job ID: {job_id if 'job_id' in locals() else 'N/A'}")
    
    # Try to get job details
    if 'job_id' in locals():
        try:
            response = requests.get(
                f"{CUSTOMIZER_URL}/v1/customization/jobs/{job_id}",
                timeout=10
            )
            if response.status_code == 200:
                job_details = response.json()
                print(f"\n📊 Job details:")
                print(json.dumps(job_details, indent=2))
        except Exception as e:
            print(f"\n⚠️  Could not fetch job details: {e}")
    
    print(f"\n" + "="*70)
    print(f"🎯 TRAINING PHASE COMPLETE")
    print(f"="*70)
    print(f"\n✅ The customized model is now stored in NeMo's model registry (Entity Store/DataStore)")
    print(f"\n📋 Next Steps - Manual Deployment:")
    print(f"   1. Copy the customized model from NeMo's registry to MinIO")
    print(f"      - Model name: {customized_model_name if customized_model_name else 'Check job details above'}")
    print(f"      - The model is stored in Entity Store/DataStore")
    print(f"      - You need to download it and upload to MinIO bucket")
    print(f"\n   2. Update or create InferenceService:")
    print(f"      - Option A: Update existing InferenceService to point to new MinIO path")
    print(f"      - Option B: Create new InferenceService with customized model path")
    print(f"\n   3. After deployment, proceed to the testing notebook:")
    print(f"      📓 test-customized-model.ipynb")
    print(f"      This notebook will test the customized model and compare with base model")
    print(f"\n💡 For detailed instructions on copying to MinIO, see Step 18.0 in the testing notebook")
else:
    print("⚠️  Customization job did not complete successfully")
    if 'job_status' in locals():
        print(f"   Job status: {job_status}")
    else:
        print(f"   Job status not available - make sure to run the wait step first")

## Step 18: Export Customized Model (Use Python Scripts)

**IMPORTANT**: EntityHandler exports customized models to DataStore after training. Use the 
Python scripts from your local machine (with port-forwards) to export model info, download from DataStore, 
merge with base model, and upload to MinIO.

### Use these scripts (from your local machine with port-forwards):

```bash
# 1. Get model info
python export_model_from_entity_store.py  # or --job-id <id>

# 2. Download from DataStore
python download_model_from_datastore.py --model-info model_info.json --output-dir ./downloaded_model

# 3. Merge adapter with base model
python merge_adapter_with_base.py --adapter-dir ./downloaded_model --output-dir ./merged_model

# 4. Upload to MinIO
MINIO_ENDPOINT=http://localhost:9000 python upload_model_to_minio.py --model-dir ./merged_model --target-path models/llama-3.2-1b-instruct-cust
```

**Note**: Use the Python scripts from your machine; they handle port-forwards and cluster access.

## Summary - Training Phase

This notebook has completed the **training phase** of the customization workflow:

1. ✅ Customizer service health (`/v1/health/ready`)
2. ✅ Customization jobs listing (`/v1/customization/jobs`)
3. ✅ DataStore service connectivity (`/v1/health`)
4. ✅ Entity Store service connectivity (`/health`)
5. ✅ DataStore namespace creation
6. ✅ Dataset upload to DataStore (via HuggingFace API)
7. ✅ Dataset registration in Entity Store
8. ✅ Verification that Customizer can find the dataset
9. ✅ Base model testing (before customization)
10. ✅ Training data preparation and upload
11. ✅ Customization dataset registration
12. ✅ Customization job creation
13. ✅ Job monitoring and completion verification

### Key Findings

**DataStore Upload**: ✅ Works via HuggingFace-compatible API (`{DATASTORE_URL}/v1/hf`)

**Entity Store Registration**: ✅ Required for Customizer to find datasets
- Customizer uses Entity Store dataset IDs, not direct DataStore paths
- Files URL format: `hf://datasets/{namespace}/{dataset_name}`

**Customizer Access**: ✅ Can access datasets via Entity Store
- Customizer queries Entity Store to resolve dataset IDs
- Entity Store provides the `hf://datasets/...` URL that points to DataStore

**Customization Workflow**: ✅ Training phase complete
- Base model testing (before customization)
- Training data preparation and upload
- Customization job creation and monitoring
- Job completion verification

### Next Steps

**Manual Deployment Required:**
1. Copy the customized model from NeMo's registry (Entity Store/DataStore) to MinIO
2. Update existing InferenceService or create new one pointing to MinIO path

**Then proceed to testing:**
- Open `test-customized-model.ipynb` to test the customized model
- Compare responses between base and customized models